# 第93课 · 让实验永不"失忆"——MLOps：W&B 实验追踪（experiment tracking）、模型版本管理与 Docker 打包

**目标**：Docker 可复现打包 + W&B 实验追踪。

> **清单体**：版本 / 监控 / 回滚——实验永不「失忆」。

🔗 Aurora 连接：`src/aurora/` 中的训练脚本 → Docker 镜像 → W&B Dashboard，实现从研究代码到可重现工程的最后一步。

← **上一课**　[L92 · 端到端流水线](L92_pipeline.ipynb)

> 上节课学习了 **端到端流水线**：麦克风 → ASR → LLM → 文本回答，全链路组装。  
> 本课将探讨 **MLOps 基础**。

## 学习目标

完成本课后，你能够：

1. **解释多阶段 Dockerfile 的体积收益**：区分 builder 阶段与 runtime 阶段，说明层缓存（layer cache）工作原理。
2. **写出完整的 W&B 实验追踪代码**：使用 `wandb.init(mode='disabled')` / `wandb.log()` / `wandb.finish()` 记录超参与指标。
3. **配置 GitHub Actions CI/CD**：理解触发条件、matrix 策略与 artifact 上传的作用，能结合项目实际 `.github/workflows/ci.yml` 进行调整。
4. **实现推理延迟监控函数**：正确计算 p50/p95 分位数并与 SLA 阈值对比，避免 off-by-one 错误。

研究代码在自己机器上跑通只是起点；真正的可重现性要求：任何人在任何机器上执行同一个镜像，能得到相同结果。
Docker 解决"环境漂移"问题，W&B 解决"实验失忆"问题——两者组合起来，让超参调优查得到记录、模型版本也追得回来。

In [ ]:
# pip install wandb docker
try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False
    print("⚠️ wandb 未安装，训练监控将跳过")

## 1. Dockerfile Best Practices：多阶段构建

**问题**：把开发依赖（pytest、black、jupyter）打进镜像会让镜像体积膨胀到 2 GB+，部署慢、攻击面大。

**解法**：多阶段构建（multi-stage build）——第一阶段 `builder` 安装所有依赖并编译，第二阶段 `runtime` 只从 `builder` 复制运行时需要的文件。

`.dockerignore` 排除 `__pycache__/`、`*.egg-info`、`.git/`、`notebooks/` 等，避免 build context 过大。

关键规则：
- `COPY pyproject.toml .` 先于 `COPY src/ src/`，利用层缓存——依赖不变时跳过 `pip install`
- 运行时用非 root 用户：`RUN useradd -m aurora && USER aurora`
- 固定基础镜像版本：`FROM python:3.11.9-slim` 而非 `python:latest`

In [ ]:
# 演示：打印一份最小可用的 Dockerfile 内容（不实际构建）
dockerfile = '''
# ── 阶段 1：builder ──────────────────────────────
FROM python:3.11.9-slim AS builder
WORKDIR /build
COPY pyproject.toml README.md ./
COPY src/ src/
RUN pip install --no-cache-dir --prefix=/install .[audio]

# ── 阶段 2：runtime ──────────────────────────────
FROM python:3.11.9-slim
COPY --from=builder /install /usr/local
RUN useradd -m aurora
USER aurora
ENTRYPOINT ["python", "-m", "aurora.serve"]  # aurora.serve 为最小入口，见 src/aurora/serve.py
'''
print(dockerfile)

# .dockerignore 示例
dockerignore = '''
__pycache__/
*.egg-info/
.git/
notebooks/
*.pyc
.env
'''
# ⚠️ ENTRYPOINT 中的 aurora.serve 已在 src/aurora/serve.py 创建最小 stub
print('--- .dockerignore ---')
print(dockerignore)

## 2. W&B 实验追踪：`wandb.init()` / `wandb.log()` / `wandb.save()`

`wandb.init(project='aurora-kws', config=cfg)` 开启一次 run，`config` 中填入所有超参（lr、batch_size、n_mels 等）。

训练循环中每步调用 `wandb.log({"loss": loss, "acc": acc, "step": step})`，W&B 自动绘制折线图。

训练结束调用 `wandb.save("model.pt")` 把权重上传到 run artifact，保证模型与超参一一对应。

三行核心代码：
```python
wandb.init(project='aurora-kws', config={'lr': 1e-3, 'batch_size': 32})
wandb.log({'loss': 0.42, 'val_acc': 0.91})
wandb.save('model.pt')
```

完整 run 结束后 `wandb.finish()` 关闭连接（Jupyter 中可以省略，脚本中建议显式调用）。

In [ ]:
# 演示：用 wandb 的 'disabled' 模式本地模拟一次训练 run（不需要联网）
import math, random

cfg = {'lr': 1e-3, 'batch_size': 32, 'epochs': 5, 'n_mels': 64}

if HAS_WANDB:
    run = wandb.init(
        project='aurora-kws-demo',
        config=cfg,
        mode='disabled',   # 离线演示，不实际上传
        name='demo-run'
    )

# 模拟训练循环
for epoch in range(cfg['epochs']):
    loss = 1.0 * math.exp(-0.6 * epoch) + random.uniform(0, 0.05)
    val_acc = 1 - 0.5 * math.exp(-0.8 * epoch) + random.uniform(-0.02, 0.02)
    if HAS_WANDB:
        wandb.log({'epoch': epoch, 'train_loss': round(loss, 4), 'val_acc': round(val_acc, 4)})
    print(f'epoch={epoch}  loss={loss:.4f}  val_acc={val_acc:.4f}')

# 模拟保存模型
# wandb.save('model.pt')  # 联网时取消注释

if HAS_WANDB:
    wandb.finish()
print('✅ W&B run 完成（disabled 模式，仅本地演示）')

## 3. 实验追踪的价值：超参对比与最优配置定位

单次实验结果没有参考价值；**可比较的多次 run** 才能回答「lr=1e-3 比 lr=1e-4 好多少」。

W&B 的 **parallel coordinates plot** 把每组超参映射到一条折线，穿过最低 val_loss 区域的参数组合即为候选最优。

W&B **Sweep** 自动化这个过程：定义搜索空间 → Sweep Controller 分配任务 → 多个 agent 并行跑实验 → Dashboard 实时汇总。

对 Aurora KWS CNN 来说，最关键的两个轴是：
- `lr`：决定收敛（convergence）速度与稳定性（搜索范围 `1e-4` 到 `1e-2`）
- `batch_size`：影响梯度噪声（gradient noise）和训练时间（候选 `16 / 32 / 64`）

In [ ]:
# 演示：打印一份 W&B Sweep 配置（YAML 等价内容，用 dict 展示）
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'lr': {
            'values': [1e-4, 1e-3, 1e-2]
        },
        'batch_size': {
            'values': [16, 32, 64]
        }
    }
}

import json as _json
print('sweep_config =')
print(_json.dumps(sweep_config, indent=2))
print()
print('grid search 共', len(sweep_config['parameters']['lr']['values'])
      * len(sweep_config['parameters']['batch_size']['values']), '组实验')

# 联网环境下执行 Sweep 的方式：
# sweep_id = wandb.sweep(sweep_config, project='aurora-kws')
# wandb.agent(sweep_id, function=train_fn, count=9)
print('✅ Sweep 配置打印完成（联网时取消注释执行）')

## 4. GitHub Actions CI/CD：每次 push 自动运行 lint + test

W&B 追踪实验指标，Docker 打包运行环境——但谁来确保代码改动不破坏已有功能？
答案是 **CI（持续集成）**：每次 `git push` 自动触发 `make lint test`，失败则阻止合并。

Aurora 的 CI 配置文件放在 `.github/workflows/ci.yml`：

```yaml
name: Aurora CI
on:
  push:
    branches: [main, "claude/**"]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - name: Install
        run: pip install -e ".[dev]"
      - name: Lint
        run: make lint
      - name: Test
        run: make test
      - name: Upload W&B summary
        if: github.ref == 'refs/heads/main'
        env: { WANDB_API_KEY: "${{ secrets.WANDB_API_KEY }}" }
        run: python scripts/log_ci_metrics.py
```

**关键点**：
- `on: push` + `pull_request` 覆盖所有常见触发场景
- `secrets.WANDB_API_KEY` 通过 GitHub Secrets 注入，不硬编码在代码里
- `log_ci_metrics.py` 把 test pass rate、lint 错误数上传到 W&B，与超参实验对齐可视化

**监控与告警**：生产部署后，需要监控推理延迟和错误率：

```python
# scripts/log_ci_metrics.py
import wandb, subprocess, json

result = subprocess.run(["pytest", "--json-report", "--json-report-file=report.json"],
                        capture_output=True)
with open("report.json") as f:
    report = json.load(f)

wandb.init(project="aurora-ci", job_type="ci")
wandb.log({
    "tests/total":   report["summary"]["total"],
    "tests/passed":  report["summary"]["passed"],
    "tests/failed":  report["summary"]["failed"],
    "pass_rate":     report["summary"]["passed"] / report["summary"]["total"],
})
wandb.finish()
```

In [ ]:
import math
import json, os, subprocess
from pathlib import Path

# ── GitHub Actions 工作流文件生成（写入 .github/workflows/ 目录）─────────────
# ⚠️ 教学示例：以下 CI YAML 与项目实际 .github/workflows/ci.yml 有差异
# 实际 CI：matrix python 3.10/3.11/3.12，所有分支触发，使用 ruff check + pytest --cov
# 如需同步，请参考 .github/workflows/ci.yml

CI_YAML = """\
name: Aurora CI

on:
  push:
    branches: [main, "claude/**"]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11"]

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Install
        run: pip install -e ".[dev]"

      - name: Lint
        run: make lint

      - name: Test with coverage
        run: |
          pip install pytest-json-report
          pytest --json-report --json-report-file=report.json -v

      - name: Upload test report
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: test-report
          path: report.json

      - name: Log to W&B
        if: github.ref == 'refs/heads/main'
        env:
          WANDB_API_KEY: ${{ secrets.WANDB_API_KEY }}
        run: python scripts/log_ci_metrics.py
"""

CI_SCRIPT = """\
#!/usr/bin/env python3
\"\"\"scripts/log_ci_metrics.py — CI 测试结果上传到 W&B\"\"\"
import json, os, wandb

with open("report.json") as f:
    report = json.load(f)

summary = report["summary"]
run = wandb.init(
    project="aurora-ci",
    job_type="ci",
    config={"python": "3.11", "branch": os.getenv("GITHUB_REF_NAME", "unknown")},
)
wandb.log({
    "tests/total":  summary["total"],
    "tests/passed": summary["passed"],
    "tests/failed": summary.get("failed", 0),
    "pass_rate":    summary["passed"] / max(summary["total"], 1),
})
wandb.finish()
print(f"✅ W&B 上传完成  pass_rate={summary['passed']}/{summary['total']}")
"""

# 打印（不实际写入，避免修改项目结构）
print("=== .github/workflows/ci.yml ===")
print(CI_YAML)
print("=== scripts/log_ci_metrics.py ===")
print(CI_SCRIPT)

# ── 监控阈值检查（可在本地 CI 脚本中直接运行）──────────────────────────────────
def check_test_quality(report_path: str = "report.json",
                       min_pass_rate: float = 0.95) -> bool:
    """读取 pytest JSON report，断言通过率 >= 阈值。"""
    if not os.path.exists(report_path):
        print(f"⚠  {report_path} 不存在，跳过检查（本地未运行 pytest）")
        return True

    with open(report_path) as f:
        report = json.load(f)
    summary = report["summary"]
    pass_rate = summary["passed"] / max(summary["total"], 1)
    ok = pass_rate >= min_pass_rate
    status = "✅" if ok else "❌"
    print(f"{status} 通过率 {pass_rate:.1%}  ({summary['passed']}/{summary['total']})  "
          f"阈值={min_pass_rate:.0%}")
    return ok

passed = check_test_quality()

# ── 简单延迟监控（生产服务用 Prometheus + Grafana，这里演示逻辑）────────────────
import time, statistics

def monitor_inference_latency(n_trials: int = 20,
                              sla_p95_ms: float = 3000.0) -> dict:
    """模拟 N 次推理，计算 p50 / p95 延迟，与 SLA 对比。"""
    import random
    random.seed(42)
    latencies = [random.gauss(2200, 200) for _ in range(n_trials)]
    latencies.sort()
    p50 = statistics.median(latencies)
    # 正确索引：ceil(0.95*N)-1，避免 off-by-one 返回最大值（p100）
    p95 = latencies[int(math.ceil(0.95 * n_trials)) - 1]
    assert p95 < latencies[-1], "p95 不应等于最大值，请检查索引"
    ok  = p95 <= sla_p95_ms
    return {"p50_ms": p50, "p95_ms": p95, "sla_ok": ok, "n": n_trials}

perf = monitor_inference_latency()
status = "✅" if perf["sla_ok"] else "❌ SLA 违约"
print(f"\n{status}  推理延迟  p50={perf['p50_ms']:.0f}ms  "
      f"p95={perf['p95_ms']:.0f}ms  SLA={3000}ms")

## 参数实验：`lr` × `batch_size` Grid Search

**搜索空间**：
- `lr` ∈ `{1e-4, 1e-3, 1e-2}` — 3 个学习率（learning rate，lr）
- `batch_size` ∈ `{16, 32, 64}` — 3 个批大小
共 9 组实验，全部 grid 遍历。

**预期现象**：
- `lr=1e-2` + `batch_size=16`：梯度噪声大 + 学习率偏高，loss 震荡，val_acc 通常最低
- `lr=1e-3` + `batch_size=32`：经验最优区间，val_acc 峰值最高
- `lr=1e-4` + `batch_size=64`：收敛最慢，5 epoch 内 val_acc 还未到顶

W&B Dashboard 的 **parallel coordinates plot** 中，穿过高 `val_acc` 区域的折线集中在 `lr≈1e-3`，可视化地验证经验规律。

In [ ]:
# 本地模拟 9 组 grid search，观察 val_acc 分布
import math, random, itertools

lrs = [1e-4, 1e-3, 1e-2]
batch_sizes = [16, 32, 64]
EPOCHS = 5

results = []

for lr, bs in itertools.product(lrs, batch_sizes):
    random.seed(int(lr * 1e5) + bs)   # 固定种子保证可复现
    noise_scale = 0.05 / (bs ** 0.5)  # 大 batch 噪声小
    lr_penalty = abs(math.log10(lr) + 3) * 0.06  # 偏离 1e-3 越远惩罚越大
    final_acc = 0.91 - lr_penalty + random.uniform(-noise_scale, noise_scale)
    final_acc = round(min(max(final_acc, 0.6), 0.99), 4)
    results.append((lr, bs, final_acc))

# 按 val_acc 降序打印
results.sort(key=lambda x: -x[2])
print(f"{'lr':>8}  {'batch_size':>10}  {'val_acc':>8}")
print('-' * 34)
for lr, bs, acc in results:
    marker = ' ← 最优' if acc == results[0][2] else ''
    print(f"{lr:>8.1e}  {bs:>10}  {acc:>8.4f}{marker}")

print()
print('✅ Grid search 模拟完成')
print(f'最优配置：lr={results[0][0]:.1e}, batch_size={results[0][1]}, val_acc={results[0][2]}')

## 5. 模型版本管理：让每次实验都可复现

每次训练都保存 `.pt` 文件很快就失控。
模型版本管理把权重文件、超参、数据集版本、代码 commit 绑定在一起。

### 最简实践：W&B Artifacts

```python
artifact = wandb.Artifact("kws-model", type="model",
                           metadata={"val_acc": 0.913, "config": config})
artifact.add_file("checkpoints/best.pt")
run.log_artifact(artifact)
# 之后还原：wandb artifact get aurora/kws-model:v3
```

### Git + Artifacts 分工

| 进 Git | 不进 Git（进 Artifacts / DVC） |
|---|---|
| 代码、配置、Dockerfile | 模型权重（> 50 MB） |
| `pyproject.toml` | 原始数据集 |
| 实验配置 YAML | W&B run 日志 |

In [ ]:
# 打印规范的 .gitignore（Aurora 场景）
lines = [
    "# 模型权重", "checkpoints/", "*.pt", "*.onnx", "*.bin", "",
    "# 数据集", "data/raw/", "data/cache/", "*.wav", "*.flac", "",
    "# W&B", "wandb/", "",
    "# Python", "__pycache__/", "*.pyc", ".venv/", "dist/", "*.egg-info/", "",
    "# Jupyter", ".ipynb_checkpoints/", "",
    "# 系统", ".DS_Store",
]
print("\n".join(lines))

## 6. ✏️ 练习：为一次 KWS 训练写完整追踪配置

In [ ]:
# ✏️ 填写你的实验配置（任何时候运行都能复现这次实验）
experiment_config = {
    "project":       "aurora-kws",
    "run_name":      None,   # 例: "mel80-lr1e3-bs32-e20"
    "dataset":       None,   # 例: "google_speech_commands_v2"
    "n_mels":        None,
    "win_len_ms":    None,
    "hop_ms":        None,
    "lr":            None,
    "batch_size":    None,
    "epochs":        None,
    "optimizer":     None,
    "best_val_acc":  None,
    "test_acc":      None,
}
filled = sum(1 for v in experiment_config.values() if v is not None)
print(f"配置完整度：{filled}/{len(experiment_config)}")
if experiment_config["run_name"]:
    print(f"Run: {experiment_config['run_name']}")

In [ ]:
# ✏️ 进阶练习：实现 setup_experiment() 函数
# 要求：接收实验配置 dict，初始化 W&B run（disabled 模式），返回 run 对象
# 验证：run.config['n_mels'] == cfg['n_mels']

def setup_experiment(cfg: dict):
    """
    初始化一次 W&B 实验 run。
    参数：
        cfg: 超参配置字典，至少包含 'project'、'run_name'、'n_mels' 键
    返回：
        wandb.Run 对象（disabled 模式，无需联网）
    """
    # TODO: 用 HAS_WANDB 守卫，在 disabled 模式下调用 wandb.init()
    # TODO: 传入 project=cfg['project'], name=cfg['run_name'], config=cfg, mode='disabled'
    # TODO: 返回 run 对象
    raise NotImplementedError('请实现 setup_experiment()')

# ── 验证（不修改以下代码）──────────────────────────────────────────
test_cfg = {
    'project': 'aurora-kws',
    'run_name': 'mel64-lr1e3-bs32-e10',
    'n_mels': 64,
    'lr': 1e-3,
    'batch_size': 32,
    'epochs': 10,
}
if HAS_WANDB:
    try:
        run = setup_experiment(test_cfg)
        assert run is not None, 'setup_experiment() 不应返回 None'
        assert run.config['n_mels'] == test_cfg['n_mels'], \
            f"n_mels 不匹配：期望 {test_cfg['n_mels']}，实际 {run.config['n_mels']}"
        assert run.config['lr'] == test_cfg['lr'], \
            f"lr 不匹配：期望 {test_cfg['lr']}，实际 {run.config['lr']}"
        run.finish()
        print('✅ setup_experiment() 验证通过')
    except (NotImplementedError, TypeError):
        print('⏳ 请实现 setup_experiment() 后重新运行')
    except AssertionError as e:
        print(f'❌ 验证失败：{e}')
else:
    print('⚠️ wandb 未安装，跳过 setup_experiment() 验证')

## 本课收束

本节把实验记录、镜像打包和版本管理串成一条生产化链路：`W&B` 负责让实验不失忆，`Docker` 负责让环境不漂移。
当训练、指标和部署都能回溯时，Aurora 的研究代码才真正走到可重现工程这一步。
下一课：**L94** Aurora v1 全景 Demo——把六个月的成果整理成一段能讲清楚、能展示、也能自证的故事。

---

→ **下一课**　[L94 · Aurora v1 全景 Demo](L94_demo.ipynb)

> 下节课将学习 **Aurora v1 全景 Demo**：综合展示所有能力，把你做的东西整理成能讲清楚的作品与思路。